<a href="https://colab.research.google.com/github/NiranjanHebli/langgraph-supervisor/blob/main/langgraph_supervisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
%pip install -q -U langgraph langchain-groq tavily-python langchain_community python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import operator
from typing import Annotated, List, TypedDict, Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
# Graph with recovery and breakpoints
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

In [13]:
from dotenv import load_dotenv
load_dotenv()  # Loads variables from .env

True

In [14]:
# The State object that travels through the graph
class AgentState(TypedDict):
    task: str
    research_notes: Annotated[List[str], operator.add]
    draft: str
    next_node: str
    retry_count: int
    revision_feedback: str

In [15]:
# Schema for the Supervisor to follow
class Router(BaseModel):
    """Decide which worker to call next."""
    next_worker: Literal["researcher", "writer", "FINISH"] = Field(description="The next node to act")
    instructions: str = Field(description="Specific instructions for the worker")
    is_critical: bool = Field(description="If True, system will pause for human review")

In [16]:
# Using Llama 3.3 70B for the "Brain" (Supervisor)
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
search_tool = TavilySearchResults(k=2)

In [17]:
def researcher(state: AgentState):
    print("🚀 Researcher is digging...")
    query = state['task']
    results = search_tool.invoke(query)
    return {"research_notes": [str(results)], "retry_count": 0}

def writer(state: AgentState):
    print("✍️ Writer is composing...")
    context = "\n".join(state['research_notes'])
    res = llm.invoke(f"Write a report on {state['task']} using: {context}")
    return {"draft": res.content}

In [18]:
def supervisor(state: AgentState):
    print("🧠 Supervisor is reviewing state...")
    # Groq supports structured output natively
    structured_llm = llm.with_structured_output(Router)

    prompt = f"""
    Task: {state['task']}
    Notes collected: {len(state['research_notes'])}
    Current Draft: {state['draft'][:100]}...
    Decide if we need more research, a better draft, or if we are done.
    If the notes are very close to the task mentioned avoid deciding research
    """

    decision = structured_llm.invoke(prompt)
    return {
        "next_node": decision.next_worker,
        "revision_feedback": decision.instructions
    }

In [19]:
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("supervisor", supervisor)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)

# Logic
builder.set_entry_point("supervisor")

builder.add_conditional_edges(
    "supervisor",
    lambda x: x["next_node"],
    {"researcher": "researcher", "writer": "writer", "FINISH": END}
)

builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")

# Enable Persistence (Recovery) and Breakpoints (Pause)
memory = MemorySaver()
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["writer"] # PAUSE: Allows human to review research before writing starts
)

In [20]:
config = {"configurable": {"thread_id": "workshop_user_1"}}
initial_input = {"task": "Impact of LPU architecture on AI inference speeds", "research_notes": [], "retry_count": 0, "draft": ""}

# 1. Start execution
print("--- STARTING GRAPH ---")
for event in graph.stream(initial_input, config, stream_mode="values"):
    if "next_node" in event:
        print(f"Moving to: {event['next_node']}")

# 2. Check if we hit a breakpoint (Pause)
snapshot = graph.get_state(config)
if snapshot.next:
    print(f"\n⏸ SYSTEM PAUSED. Next step is: {snapshot.next}")
    print(f"Feedback from Supervisor: {snapshot.values['revision_feedback']}")

# 3. Resume (Human says 'Go ahead')
print("\n--- RESUMING AFTER PAUSE ---")
for event in graph.stream(None, config, stream_mode="values"):
    print("Finalizing...")
    print(event["draft"])

--- STARTING GRAPH ---
🧠 Supervisor is reviewing state...
Moving to: researcher
🚀 Researcher is digging...
Moving to: researcher
🧠 Supervisor is reviewing state...
Moving to: writer

⏸ SYSTEM PAUSED. Next step is: ('writer',)
Feedback from Supervisor: Create a draft based on the collected notes about the impact of LPU architecture on AI inference speeds

--- RESUMING AFTER PAUSE ---
Finalizing...

✍️ Writer is composing...
Finalizing...
**Report: Impact of LPU Architecture on AI Inference Speeds**

**Introduction**

The increasing demand for real-time responsiveness and expedited language processing capabilities has led to the development of Language Processing Units (LPUs). LPUs are purpose-built hardware architectures designed specifically for Large Language Model (LLM) inference, aiming to overcome the limitations of traditional GPU-based approaches. This report explores the impact of LPU architecture on AI inference speeds, highlighting its benefits, performance advantages, and des